# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding: "Which Pages Will Grow?" (growth prediction model)**

The paper reports 90% accuracy on same-brand held-out pages and 75% on entirely unseen brands, 
trained on 96.6K pages "clearly growing or declining."

*My methodology question:* Where does the "growing vs declining" label come from, and over what 
window? The paper doesn't specify whether the label is defined from the same window as the features 
(e.g. last-30-days trend used both to build features like "Days Visible" and to define the label), 
which is exactly the leakage risk I ran into myself in w03. If the label and features share any time 
window, the 90%/75% numbers could be inflated the same way my own leaked-feature demo jumped to 0.998. 
I'd want to see the exact label definition and confirm the feature window ends strictly before the 
label window begins — the same discipline the report asks readers to bring to their own work 
("Content age confounds model comparisons" is flagged, but I don't see the same caution applied to 
label/feature timing here).

**Finding: "The Freshness Multiplier" — 361+ day bucket (37.19:1 ratio)**

The paper itself flags this: the 361+ freshness bucket shows a 37.19:1 growth-to-decline ratio, but 
it's based on only 802 pages with just 21 declining pages total.

*My methodology question:* Does the validation design support featuring this number as prominently 
as it is? To the paper's credit, it explicitly warns "don't read too much into it" in the chart-read 
note — but the ratio still appears as a headline card (37.2x) right next to the more stable 5.43:1 
number from the 31-90 day bucket (n=18.8K). With only 21 declining pages, a handful of unusual cases 
could swing this ratio dramatically. I'd ask: would this finding survive a bootstrap confidence 
interval, or does it disappear at that sample size? This is a good example of a genuinely honest 
disclosure (the paper does flag it) that I think could be even more careful by not giving the small-n 
number equal visual weight to the stable one.

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = df[df["impressions_90d"] >= 100].copy()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

feature_cols = [
    "avg_position", "ctr", "impressions_90d", "sessions_90d",
    "word_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition"
]
X = df[feature_cols].fillna(0)
y = df["is_declining_label"]

def precision_at_k(y_true, scores, k=50):
    top_k_idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_idx].sum() / k

# BEFORE: naive random split (row-level, ignores client grouping)
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler_naive = StandardScaler()
Xtr_n = scaler_naive.fit_transform(X_train_naive)
Xte_n = scaler_naive.transform(X_test_naive)
model_naive = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_n, y_train_naive)
proba_naive = model_naive.predict_proba(Xte_n)[:, 1]
p50_naive = precision_at_k(y_test_naive.reset_index(drop=True), proba_naive, k=50)

# AFTER: client-grouped split (honest, same as w05)
groups = df["client_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
scaler_g = StandardScaler()
Xtr_g = scaler_g.fit_transform(X_train_g)
Xte_g = scaler_g.transform(X_test_g)
model_g = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_g, y_train_g)
proba_g = model_g.predict_proba(Xte_g)[:, 1]
p50_grouped = precision_at_k(y_test_g.reset_index(drop=True), proba_g, k=50)

comparison = pd.DataFrame({
    "Split type": ["Naive random split (BEFORE)", "Client-grouped split (AFTER)"],
    "Precision@50": [p50_naive, p50_grouped]
})
comparison

,Split type,Precision@50
0,Naive random split (BEFORE),0.86
1,Client-grouped split (AFTER),0.88


## 2. My model under an honest split (before/after)

| Split type | Precision@50 |
|---|---:|
| Naive random split (BEFORE) | 0.86 |
| Client-grouped split (AFTER) | 0.88 |

**Result:** Surprisingly, the honest client-grouped split scored *slightly higher* (0.88) than the 
naive random split (0.86) — the opposite of what I expected going in. I assumed the naive split would 
be inflated by client-level leakage (the model "memorizing" a client it had already partly seen), 
which would show up as a drop when I switched to the honest split.

**Why I think this happened:** with only 30 total clients in this dataset, both the naive and grouped 
splits likely end up seeing a similar mix of clients either way — the naive random split still 
scatters rows from most clients into both train and test, so there may not be much room for the model 
to "memorize" a client-specific pattern that a grouped split would then remove. The difference here 
(0.86 vs 0.88) is small enough that I'd treat it as within normal run-to-run variation rather than 
strong evidence of leakage in either direction.

**The honest takeaway:** I'm reporting both numbers rather than picking the one that supports a 
cleaner story. The grouped split remains my chosen validation design going forward — not because it 
produced a better number here, but because it's the more defensible design for the real-world 
question (scoring pages for clients, including ones outside the training set), regardless of which 
split happened to score higher on this particular run.

In [4]:
# Re-check: does any feature correlate suspiciously highly with the label?
import pandas as pd

audit_df = X.copy()
audit_df["label"] = y

correlations = audit_df.corr()["label"].drop("label").sort_values(key=abs, ascending=False)
correlations

content_age_days         -0.206320
word_count                0.127479
avg_position             -0.113769
ctr                      -0.084567
sessions_90d             -0.067048
impressions_90d          -0.063502
days_since_last_update    0.034675
search_volume            -0.025550
competition              -0.005484
Name: label, dtype: float64

## 3. Leakage audit

**Correlation check** (same hunt as w03, applied to my final w05 feature set):

| Feature | Correlation with label |
|---|---:|
| `content_age_days` | -0.206 |
| `word_count` | +0.127 |
| `avg_position` | -0.114 |
| `ctr` | -0.085 |
| `sessions_90d` | -0.067 |
| `impressions_90d` | -0.064 |
| `days_since_last_update` | +0.035 |
| `search_volume` | -0.026 |
| `competition` | -0.005 |

**Verdict: No leakage detected.** All correlations are moderate (highest is -0.206 for 
`content_age_days`). None are anywhere near the 0.9+ range I saw in my deliberate w03 leakage demo, 
where a feature built directly from the label scored 0.998. This is reassuring, but not a complete 
leakage audit on its own — correlation only catches *linear* relationships, so I also manually 
re-checked each feature's source:

- All 9 features (`avg_position`, `ctr`, `impressions_90d`, `sessions_90d`, `word_count`, 
  `content_age_days`, `days_since_last_update`, `search_volume`, `competition`) come from the same 
  90-day observation window as the label — none reference a future window relative to the label.
- None of these are FlyRank product-computed flags (`health_score`, `priority_score`, `action_type`) 
  — all are raw observed signals.
- The one caveat I'm carrying forward from w02/w03: because both the features AND the label 
  (`trend_direction == "down"`) come from the *same* 90-day window, this isn't a true forward-looking 
  prediction — it's a same-window classification. That's not leakage in the technical sense (nothing 
  from the label directly appears as a feature), but it does mean the model is learning "what does a 
  currently-declining page look like" rather than "what predicts future decline," which limits how 
  the model's claims should be framed (see Section 4).

## 4. Claim rewrite

**My boldest original claim (from w05):** "Logistic Regression is my best method here (0.88), 
slightly beating my own w04 baseline rule (0.82)."

**Why this needs softer language:** "beating" implies a settled, general fact. But this result is: 
(a) from one specific test split of 6 clients, (b) using a same-window proxy label rather than a true 
future outcome, and (c) I've since shown in Section 2 that switching the split design changes the 
number slightly (0.86 vs 0.88) — so the exact ranking between methods could shift with a different 
random seed or a different set of held-out clients.

**Rewritten in safe language:** "In this test split, Logistic Regression showed a directional 
advantage over the baseline rule (observed Precision@50 of 0.88 vs 0.82), though this comparison used 
a same-window proxy label rather than a verified future outcome, and the gap was small enough that I 
would not treat it as a settled result without testing across multiple splits. This is decision- 
support evidence for choosing Logistic Regression as a starting point, not proof that it will 
outperform the baseline in every future evaluation."

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.